## Evaluation Script

In [ ]:
from __future__ import annotations

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from scipy.optimize import curve_fit
from scipy.stats import lognorm

### Get the Data

This gets the data and prints some facts about it.

In [ ]:
results_df = pd.read_csv(
    "../data/output_data/en_wiki/step_list/en_wiki_step_list_wikigame-gpt-4o_0.tsv",
    sep="\t",
)

In [ ]:
results_df.head()

In [ ]:
sucessful_paths = results_df["num_steps"][(results_df["found_target"])]
print(f"It was sucessful {sucessful_paths.size/len(results_df):.2%} of the time")

In [ ]:
if len(sucessful_paths) > len(results_df) // 2:
    print(
        f"The median path was "
        f"{results_df['num_steps'].sort_values().iloc[len(results_df)//2]}"
    )
else:
    print("The majority of the paths failed so the median is a fail.")

In [ ]:
print("Only considering the sucessful paths")
print(f"The mean number of steps was {sucessful_paths.mean():.2f}")
print(f"The median number of steps was {sucessful_paths.median():.1f}")

### Fitting the Histograms

In [ ]:
def vals_from_lognorm(xs, mu, scalex, scaley):
    """This function is used to generate the y values for a lognormal distribution."""
    return scaley / scalex * lognorm.pdf(xs / scalex, mu) * 2

In [ ]:
Y, bins = np.histogram(sucessful_paths, bins=np.arange(0, 101, 2) + 0.5)
X = (bins[:-1] + bins[1:]) / 2

In [ ]:
fit_res = curve_fit(
    vals_from_lognorm, X, Y, [1, 1, 100], bounds=([0, 0, 1], [2, 100, 5000])
)

print(
    f"The spread metric of the lognormal distribution is "
    f"{fit_res[0][0]:.4g} +/- {np.sqrt(fit_res[1][0][0]):.3g}"
)

print(
    f"The median steps of the lognormal distribution is "
    f"{fit_res[0][1]:.4g} +/- {np.sqrt(fit_res[1][1][1]):.3g} "
    f"steps per game."
)

print(
    f"The total area of the lognormal distribution is "
    f"{fit_res[0][2]:.4g} +/- {np.sqrt(fit_res[1][2][2]):.3g} "
    f"successful games at infinity."
)

In [ ]:
x_p = np.arange(0.5, 101, 0.5)

plt.figure(figsize=(6, 4))

plt.hist(sucessful_paths, bins=bins)
plt.plot(
    x_p,
    vals_from_lognorm(x_p, fit_res[0][0], fit_res[0][1], fit_res[0][2]),
    "--k",
    lw=3,
    label="Lognormal fit",
)

plt.xlabel("Number of steps")
plt.ylabel("Frequency (binned in pairs)")

plt.title("GPT-4o - 100 games")

plt.legend(
    title=(
        f"Median - {fit_res[0][1]:.4g} +/- {np.sqrt(fit_res[1][1][1]):.3g}\n"
        f"Total - {fit_res[0][2]:.4g} +/- {np.sqrt(fit_res[1][2][2]):.3g}\n"
        f"Spread - {fit_res[0][0]:.3g} +/- {np.sqrt(fit_res[1][0][0]):.2g}"
    )
)

plt.show()